# 最初のモデル — 線形/ロジスティック回帰・決定木

基本となる線形回帰・ロジスティック回帰・決定木の特徴を理解し、Titanic でベースラインを作ります。

> **Kaggle Tutorial** 対応ノートブック（`/learn/ml-basics/08-first-models`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# Kaggle Notebook 上では既存の /kaggle/input/titanic をそのまま使用し、
# Google Colab やローカル環境では互換スキーマの Titanic サンプルデータ（891行 / 418行）を自動配置します。
# ※ Kaggle 本番データで提出する場合は kaggle CLI (!kaggle competitions download -c titanic) をご利用ください。
import os
import numpy as np
import pandas as pd

def _ensure_titanic_data():
    target_dir = "/kaggle/input/titanic"
    if os.path.exists(os.path.join(target_dir, "train.csv")):
        return

    try:
        os.makedirs(target_dir, exist_ok=True)
        data_dir = target_dir
    except OSError:
        data_dir = os.path.abspath("./.kaggle_input/titanic")
        os.makedirs(data_dir, exist_ok=True)
        _orig_read_csv = pd.read_csv

        def _compat_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/"):
                rel = filepath_or_buffer[len("/kaggle/input/"):]
                local_path = os.path.abspath(os.path.join("./.kaggle_input", rel))
                if os.path.exists(local_path):
                    filepath_or_buffer = local_path
            return _orig_read_csv(filepath_or_buffer, *args, **kwargs)

        pd.read_csv = _compat_read_csv

    rng = np.random.default_rng(42)
    surnames = ["Braund", "Cumings", "Heikkinen", "Futrelle", "Allen", "Moran", "McCarthy", "Palsson", "Johnson", "Nasser"]
    given_m = ["Owen", "William", "James", "Henry", "Leonard", "Timothy", "charles", "George"]
    given_f = ["Florence", "Laina", "Lily", "elisabeth", "Adele", " Marguerite", "Anna", "Mary"]

    def _make_split(n, start_id, with_target):
        pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])
        sex = rng.choice(["male", "female"], size=n, p=[0.65, 0.35])
        age = np.clip(rng.normal(29.7, 14.5, size=n), 0.42, 80.0).round(1)
        sibsp = rng.choice([0, 1, 2, 3, 4], size=n, p=[0.68, 0.23, 0.05, 0.02, 0.02])
        parch = rng.choice([0, 1, 2, 3], size=n, p=[0.76, 0.13, 0.09, 0.02])
        base_fare = np.where(pclass == 1, 60.0, np.where(pclass == 2, 20.0, 11.0))
        fare = (base_fare * rng.lognormal(0.0, 0.55, size=n)).round(2)
        embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.72, 0.19, 0.09]).astype(object)

        names = []
        cabins = []
        tickets = []
        for i in range(n):
            sn = surnames[i % len(surnames)]
            if sex[i] == "male":
                title = "Master" if age[i] < 13 else "Mr"
                gn = given_m[i % len(given_m)].strip().capitalize()
            else:
                title = "Miss" if age[i] < 24 else "Mrs"
                gn = given_f[i % len(given_f)].strip().capitalize()
            names.append(f"{sn}, {title}. {gn}")
            cabins.append(f"{'ABC'[pclass[i] - 1]}{10 + (i % 90)}" if rng.random() > 0.77 else np.nan)
            tickets.append(f"PC {17500 + (i % 350)}" if pclass[i] == 1 else str(230000 + i))

        age_obj = age.astype(float)
        age_obj[rng.random(n) < 0.20] = np.nan
        embarked[rng.random(n) < 0.005] = np.nan
        if not with_target:
            fare[0] = np.nan

        data = {
            "PassengerId": np.arange(start_id, start_id + n),
            "Pclass": pclass,
            "Name": names,
            "Sex": sex,
            "Age": age_obj,
            "SibSp": sibsp,
            "Parch": parch,
            "Ticket": tickets,
            "Fare": fare,
            "Cabin": pd.Series(cabins, dtype=object),
            "Embarked": pd.Series(embarked, dtype=object),
        }
        if with_target:
            family_size = sibsp + parch + 1
            logit = (
                -1.65
                + np.where(sex == "female", 2.45, 0.0)
                + np.where(pclass == 1, 1.35, np.where(pclass == 2, 0.35, -0.55))
                + np.where(np.nan_to_num(age_obj, nan=29.0) < 12, 1.2, 0.0)
                + np.where((family_size >= 2) & (family_size <= 4), 0.75, -0.35)
            )
            prob = 1.0 / (1.0 + np.exp(-logit))
            survived = (rng.random(n) < prob).astype(int)
            ordered = {"PassengerId": data["PassengerId"], "Survived": survived}
            for k, v in data.items():
                if k != "PassengerId":
                    ordered[k] = v
            return pd.DataFrame(ordered)
        return pd.DataFrame(data)

    _make_split(891, 1, True).to_csv(os.path.join(data_dir, "train.csv"), index=False)
    _make_split(418, 892, False).to_csv(os.path.join(data_dir, "test.csv"), index=False)
    print(f"Titanic dataset ready at: {data_dir}")

_ensure_titanic_data()

## 3 つの基本モデル

| モデル                     | タスク    | 特徴                                                   |
| -------------------------- | --------- | ------------------------------------------------------ |
| **線形回帰**           | 回帰      | 特徴量の線形結合で連続値を予測。解釈しやすい           |
| **ロジスティック回帰** | 分類      | 線形結合を確率に変換。二値分類の定番ベースライン       |
| **決定木**             | 回帰/分類 | 条件分岐で予測。非線形を扱えるが単体では過学習しやすい |

まずはシンプルなモデルで**ベースライン**を作り、以降の改善の基準にします。

## Titanic でベースラインを作る

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

df = pd.read_csv("/kaggle/input/titanic/train.csv")
features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
X = df[features]
y = df["Survived"]

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
print(f"CV accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")

## 決定木と過学習

決定木は木を深くするほど訓練データに適合しますが、深すぎると過学習します。
`max_depth` などのハイパーパラメータで複雑さを抑えるのが基本です。

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=4, random_state=42)

In [ ]:
# 決定木でも同様に Pipeline を組んで CV スコアを比較します
tree_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", tree)])
tree_scores = cross_val_score(tree_pipe, X, y, cv=cv, scoring="accuracy")
print(f"DecisionTree CV accuracy: {tree_scores.mean():.4f} +/- {tree_scores.std():.4f}")

> **Tip:** 決定木は単体では弱めですが、これを多数束ねたランダムフォレストや勾配ブースティング （次トラック L5）が、表形式コンペの主力になります。今回はその土台を作りました。

## このトラックのまとめ

- 機械学習はデータから規則性を学ぶ技術で、Kaggle の中心は教師あり学習
- 目的変数の型でタスク（回帰/分類）と評価指標が決まる
- 汎化を測るためにデータを分割し、交差検証で安定評価する
- 指標はコンペに合わせて選び、Pipeline でリークなく前処理する

## 次のステップ

次トラック（L3）では、Titanic / House Prices を題材に、ここまでの基礎を使って
**実際のコンペに提出するまで**の流れを体験します。